In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
CSV_PATH = './data/iris.csv'

DEVICE = 'cpu'
if torch.cuda.is_available():
    DEVICE = 'cuda'

BATCH_SIZE = 16
EPOCHS = 1000
LEARNING_RATE = 0.01
SEED = 777

torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

In [3]:
df = pd.read_csv(CSV_PATH)
df

,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [4]:
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
target = 'class'

X = df[features]
Y = df[target]

In [5]:
X

,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,Y,
    test_size=0.2,
    stratify = Y, # 타겟 비율보고 나누어라
    random_state=SEED
)

In [7]:
X_train

,sepal_length,sepal_width,petal_length,petal_width
121,5.6,2.8,4.9,2.0
52,6.9,3.1,4.9,1.5
112,6.8,3.0,5.5,2.1
40,5.0,3.5,1.3,0.3
21,5.1,3.7,1.5,0.4
...,...,...,...,...
27,5.2,3.5,1.5,0.2
0,5.1,3.5,1.4,0.2
5,5.4,3.9,1.7,0.4
48,5.3,3.7,1.5,0.2


In [8]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_std = scaler.transform(X_train)
X_test_std = scaler.transform(X_test)

In [9]:
X_train_std

array([[-3.09901904e-01, -6.30204066e-01,  6.50092771e-01,
         1.04374928e+00],
       [ 1.27517013e+00,  3.90391899e-02,  6.50092771e-01,
         3.90725749e-01],
       [ 1.15324151e+00, -1.84041895e-01,  9.90504986e-01,
         1.17435399e+00],
       [-1.04147361e+00,  9.31363531e-01, -1.39238052e+00,
        -1.17653074e+00],
       [-9.19544993e-01,  1.37752570e+00, -1.27890978e+00,
        -1.04592603e+00],
       [ 1.03131289e+00,  3.90391899e-02,  3.66415926e-01,
         2.60121042e-01],
       [-1.87973286e-01,  1.60060679e+00, -1.16543904e+00,
        -1.17653074e+00],
       [ 6.65527039e-01, -4.07122981e-01,  3.09680556e-01,
         1.29516334e-01],
       [ 2.25059907e+00, -1.84041895e-01,  1.33091720e+00,
         1.43556340e+00],
       [-9.19544993e-01,  1.60060679e+00, -1.22217441e+00,
        -1.30713544e+00],
       [-1.87973286e-01, -6.30204066e-01,  1.96209818e-01,
         1.29516334e-01],
       [ 9.09384275e-01, -1.84041895e-01,  3.66415926e-01,
      

In [10]:
X_train_std = torch.tensor(X_train_std, dtype=torch.float32).to(DEVICE)
X_test_std = torch.tensor(X_test_std, dtype=torch.float32).to(DEVICE)
Y_train = torch.tensor(Y_train.values, dtype=torch.long).to(DEVICE)
Y_test = torch.tensor(Y_test.values, dtype=torch.long).to(DEVICE)

In [11]:
# Dataset
train_dataset = TensorDataset(X_train_std, Y_train)

In [12]:
for i in range(0, 10):
    print(train_dataset[i])

(tensor([-0.3099, -0.6302,  0.6501,  1.0437]), tensor(2))
(tensor([1.2752, 0.0390, 0.6501, 0.3907]), tensor(1))
(tensor([ 1.1532, -0.1840,  0.9905,  1.1744]), tensor(2))
(tensor([-1.0415,  0.9314, -1.3924, -1.1765]), tensor(0))
(tensor([-0.9195,  1.3775, -1.2789, -1.0459]), tensor(0))
(tensor([1.0313, 0.0390, 0.3664, 0.2601]), tensor(1))
(tensor([-0.1880,  1.6006, -1.1654, -1.1765]), tensor(0))
(tensor([ 0.6655, -0.4071,  0.3097,  0.1295]), tensor(1))
(tensor([ 2.2506, -0.1840,  1.3309,  1.4356]), tensor(2))
(tensor([-0.9195,  1.6006, -1.2222, -1.3071]), tensor(0))


In [13]:
# Data Loader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [14]:
batch_count = 0
for minibatch in train_loader:
    features, targets = minibatch
    print('Feature:', features)
    print('Target:', targets)
    print()

    batch_count += 1
    if batch_count >=2:
        break

Feature: tensor([[ 0.6655,  0.0390,  0.9905,  0.7825],
        [ 0.1778, -0.1840,  0.5934,  0.7825],
        [-1.4073,  0.2621, -1.3924, -1.3071],
        [-1.7730, -0.4071, -1.3356, -1.3071],
        [-1.1634,  1.1544, -1.3356, -1.4377],
        [ 0.2997, -0.6302,  0.1395,  0.1295],
        [-0.4318, -1.5225, -0.0307, -0.2623],
        [-1.5292,  0.7083, -1.3356, -1.1765],
        [ 0.0559, -0.1840,  0.2529,  0.3907],
        [-0.5538,  1.8237, -1.1654, -1.0459],
        [-1.1634, -1.2994,  0.4232,  0.6519],
        [ 0.1778, -1.9687,  0.7068,  0.3907],
        [-0.3099, -0.6302,  0.6501,  1.0437],
        [-0.1880,  1.6006, -1.1654, -1.1765],
        [ 0.4217, -1.9687,  0.4232,  0.3907],
        [ 0.7875, -0.1840,  1.1607,  1.3050]])
Target: tensor([2, 2, 0, 0, 0, 1, 1, 0, 1, 0, 2, 2, 2, 0, 1, 2])

Feature: tensor([[ 1.0313,  0.0390,  1.0472,  1.5662],
        [ 1.0313, -0.1840,  0.7068,  0.6519],
        [ 0.6655, -0.6302,  1.0472,  1.1744],
        [-0.9195,  1.3775, -1.2789, -1.04

In [15]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 10)
        self.leakyrelu = nn.LeakyReLU()
        self.fc2 = nn.Linear(10, 3)

    def forward(self, x):
        x = self.fc1(x)
        x = self.leakyrelu(x)
        x = self.fc2(x)
        return x

In [16]:
model = Classifier().to(DEVICE)

In [17]:
criterion = nn.CrossEntropyLoss().to(DEVICE)
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [18]:
model.train()
for epoch in range(EPOCHS):
    for X_batch, Y_batch in train_loader:

        X_batch = X_batch.to(DEVICE)
        Y_batch = Y_batch.to(DEVICE)

        optimizer.zero_grad()
        Y_hat = model.forward(X_batch)
        loss = criterion(Y_hat, Y_batch)
        loss.backward()
        optimizer.step()

    if (epoch+1) % 100 ==0:
        print(f'Epoch {epoch + 1}, Loss: {loss.item():.4f}')


Epoch 100, Loss: 0.0705
Epoch 200, Loss: 0.1625
Epoch 300, Loss: 0.0766
Epoch 400, Loss: 0.0384
Epoch 500, Loss: 0.0342
Epoch 600, Loss: 0.1071
Epoch 700, Loss: 0.0145
Epoch 800, Loss: 0.0392
Epoch 900, Loss: 0.2013
Epoch 1000, Loss: 0.0725


In [19]:
Y_hat

tensor([[ 8.4173,  0.8903, -7.7900],
        [-6.3446,  2.9005,  1.7571],
        [-6.3511,  0.9785,  3.2470],
        [-5.6851,  1.1021,  2.8502],
        [-7.5485, -0.1218,  5.2434],
        [-8.3665, -0.8404,  6.4620],
        [-2.5375,  3.1655, -1.3218],
        [-3.2618,  3.0576, -0.7326]], grad_fn=<AddmmBackward0>)

In [20]:
Y_batch

tensor([0, 1, 2, 2, 2, 2, 1, 1])

In [21]:
model.eval()
with torch.no_grad():
    y_pred_probability = model.forward(X_test_std)
    y_pred_label = torch.argmax(y_pred_probability, 1)
    accuracy = (y_pred_label == Y_test).sum().item() / len(Y_test)
    print('Accuracy:', accuracy)

Accuracy: 0.9333333333333333


In [22]:
y_pred_probability

tensor([[-0.7648,  3.1732, -2.8974],
        [-1.0110,  2.9180, -2.3581],
        [-1.2064,  3.0153, -2.2858],
        [-3.3605,  3.3701, -0.8590],
        [-3.9584,  2.6394,  0.2117],
        [-7.6400,  0.7247,  4.5539],
        [ 7.2211,  1.0124, -6.9449],
        [ 6.4401,  1.1313, -6.4468],
        [ 7.2277,  1.0379, -6.9888],
        [-4.8502,  0.9403,  2.2754],
        [-5.1028,  3.8264,  0.0475],
        [-6.3711,  1.3917,  3.0848],
        [-7.8649, -0.8214,  5.9122],
        [-7.0581,  0.1616,  4.6638],
        [-6.9675, -0.2175,  4.7636],
        [-3.8035,  2.9710, -0.0840],
        [-8.1776, -2.3078,  7.2338],
        [ 7.3379,  0.9782, -6.9925],
        [ 7.3338,  1.1492, -7.2544],
        [ 7.5323,  1.0124, -7.2142],
        [-7.2383, -0.0140,  4.9497],
        [-8.3075, -1.3623,  6.8291],
        [ 6.6992,  1.0743, -6.5872],
        [-6.5941,  1.2340,  3.3831],
        [-8.0705,  0.2329,  5.3024],
        [ 6.3118,  1.1029, -6.2926],
        [-6.4897,  0.4875,  3.9841],
 

In [27]:
print(torch.argmax(y_pred_probability, 1))
print(torch.argmax(y_pred_probability, 0))
print((y_pred_label == Y_test).sum().item())

tensor([1, 1, 1, 1, 1, 2, 0, 0, 0, 2, 1, 2, 2, 2, 2, 1, 2, 0, 0, 0, 2, 2, 0, 2,
        2, 0, 2, 0, 0, 1])
tensor([19, 29, 16])
28


In [28]:
from sklearn.metrics import confusion_matrix
confusion_matrix = confusion_matrix(Y_test, y_pred_label)
print(confusion_matrix)

[[10  0  0]
 [ 0  8  2]
 [ 0  0 10]]


In [29]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(Y_test, y_pred_label)
print(accuracy)

0.9333333333333333


In [33]:
from sklearn.metrics import precision_score
precision = precision_score(Y_test, y_pred_label, average='macro')
print(precision)

0.9444444444444445


In [34]:
from sklearn.metrics import recall_score
recall = recall_score(Y_test, y_pred_label, average = 'macro')
print(recall)

0.9333333333333332


In [35]:
from sklearn.metrics import f1_score
f1 = f1_score(Y_test, y_pred_label, average='macro')
print(f1)

0.9326599326599326


In [37]:
# 모델 저장하는 함수
torch.save(model.state_dict(), './iris_model01.pth')

In [38]:
# scaler 저장
import joblib
joblib.dump(scaler, './std_scaler.joblib')

['./std_scaler.joblib']